# Day 3 Lab: Bayesian Yield-Curve Risk & Scenario Analysis
You will build a probabilistic model that links Bloomberg yield-curve points with macro shocks, then communicate the uncertainty via fan charts and scenario tables.

> **Learning outcomes**
> - prepare term-structure datasets for PyMC modelling
> - encode priors reflecting policy constraints
> - run posterior predictive checks
> - translate Bayesian output into stress scenarios

## Data expectations
- Export from `YC <GO>` → `Actions > Export > CSV` with columns `date`, `tenor`, `yield` for key maturities (1M, 3M, 6M, 1Y, 2Y, 5Y, 10Y, 30Y).
- Optional macro join: `ECST <GO>` (GDP surprise, CPI surprise) or `FOMC <GO>` event dummy.
- Save as `yield_curve_YYYYMMDD.csv`; macro as `macro_surprises.csv`.

In [ ]:
%%capture
!pip install pandas numpy pymc==5.12.0 arviz plotly==5.24.0

In [ ]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import plotly.express as px

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

DATA_ROOT = Path("/content/data") if IN_COLAB else Path.cwd() / "data" / "bloomberg"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    drive.mount("/content/drive", force_remount=True)


In [ ]:
yc_path = DATA_ROOT / "yield_curve_SAMPLE.csv"
macro_path = DATA_ROOT / "macro_surprises_SAMPLE.csv"

try:
    yc = pd.read_csv(yc_path, parse_dates=["date"])
    macro = pd.read_csv(macro_path, parse_dates=["date"])
    print("Loaded Bloomberg curves + macro surprises.")
except FileNotFoundError:
    print("Files not found. Creating synthetic but realistic curve data...")
    dates = pd.date_range("2018-01-31", periods=48, freq="M")
    tenors = ["1M", "3M", "6M", "1Y", "2Y", "5Y", "10Y", "30Y"]
    rng = np.random.default_rng(0)
    base_curve = {
        "1M": 1.2,
        "3M": 1.3,
        "6M": 1.4,
        "1Y": 1.5,
        "2Y": 1.7,
        "5Y": 2.0,
        "10Y": 2.2,
        "30Y": 2.3
    }
    rows = []
    for date in dates:
        level_shift = rng.normal(0, 0.3)
        slope_shift = rng.normal(0, 0.1)
        curvature_shift = rng.normal(0, 0.05)
        for tenor, base in base_curve.items():
            maturity_years = {
                "1M": 1/12, "3M": 0.25, "6M": 0.5, "1Y": 1,
                "2Y": 2, "5Y": 5, "10Y": 10, "30Y": 30
            }[tenor]
            yield_rate = base + level_shift + slope_shift * (maturity_years - 5) / 10 + curvature_shift * ((maturity_years - 5) ** 2) / 25
            rows.append({"date": date, "tenor": tenor, "yield": yield_rate})
    yc = pd.DataFrame(rows)
    macro = pd.DataFrame({
        "date": dates,
        "gdp_surprise": rng.normal(0, 0.5, len(dates)),
        "inflation_surprise": rng.normal(0, 0.3, len(dates)),
        "fomc_hike": rng.binomial(1, 0.2, len(dates))
    })

yc = yc.merge(macro, on="date", how="left")
yc.head()

In [ ]:
yc["tenor_code"] = yc["tenor"].astype("category")
yc["tenor_idx"] = yc["tenor_code"].cat.codes
maturity_years = {
    "1M": 1/12, "3M": 0.25, "6M": 0.5, "1Y": 1,
    "2Y": 2, "5Y": 5, "10Y": 10, "30Y": 30
}
yc["maturity_years"] = yc["tenor"].map(maturity_years)

coords = {
    "observation": np.arange(len(yc)),
    "tenor": yc["tenor_code"].cat.categories
}

yields_obs = yc["yield"].values
macro_feat = yc[["gdp_surprise", "inflation_surprise", "fomc_hike"]].values
maturity_idx = yc["tenor_idx"].values

In [ ]:
with pm.Model(coords=coords) as curve_model:
    tenor_idx = pm.Data("tenor_idx", maturity_idx, dims="observation")
    macro_data = pm.Data("macro", macro_feat, dims=("observation", "macro_feature"))

    alpha = pm.Normal("alpha", mu=1.5, sigma=0.5)
    tenor_offset = pm.Normal("tenor_offset", mu=0.0, sigma=0.3, dims="tenor")
    beta_macro = pm.Normal("beta_macro", mu=0.0, sigma=0.2, dims=("macro_feature",))
    sigma = pm.Exponential("sigma", 5.0)

    mu = alpha + tenor_offset[tenor_idx] + pm.math.dot(macro_data, beta_macro)
    pm.Normal("yields", mu=mu, sigma=sigma, observed=yields_obs, dims="observation")

    curve_trace = pm.sample(1500, tune=1500, target_accept=0.92, chains=2, progressbar=True)
    posterior_predictive = pm.sample_posterior_predictive(curve_trace)
curve_trace

In [ ]:
az.summary(curve_trace, var_names=["alpha", "beta_macro", "sigma"])

In [ ]:
ppc = posterior_predictive["yields"]
ppc_mean = ppc.mean(axis=0)
ppc_low = np.percentile(ppc, 10, axis=0)
ppc_high = np.percentile(ppc, 90, axis=0)
fan_df = yc[["date", "tenor"]].copy()
fan_df["mean"] = ppc_mean
fan_df["low"] = ppc_low
fan_df["high"] = ppc_high

fig = px.area(fan_df, x="date", y="mean", color="tenor", title="Posterior fan chart (median)")
fig.show()

### Scenario builder
Input a hypothetical macro shock (GDP surprise, inflation surprise, FOMC hike) to simulate yield outcomes.

In [ ]:
import numpy as np

def simulate_scenario(gdp_surprise: float, inflation_surprise: float, fomc_hike: int = 0):
    macro_vector = np.array([gdp_surprise, inflation_surprise, fomc_hike])
    beta = curve_trace.posterior["beta_macro"].stack(draw=("chain", "draw"))
    alpha_samples = curve_trace.posterior["alpha"].stack(draw=("chain", "draw"))
    tenor_offset_samples = curve_trace.posterior["tenor_offset"].stack(draw=("chain", "draw"))
    mu = alpha_samples + (beta * macro_vector).sum("macro_feature")
    mu = mu + tenor_offset_samples
    summary = mu.to_dataframe().groupby("tenor").quantile([0.1, 0.5, 0.9]).unstack()
    summary.columns = ["p10", "p50", "p90"]
    return summary

simulate_scenario(gdp_surprise=-1.0, inflation_surprise=0.6, fomc_hike=1)

### Deliverable
- Paste the fan chart and scenario table into your day-3 slide.
- Write a paragraph explaining how Bayesian credible intervals influence treasury allocation decisions.
- Include at least one policy-aligned prior choice (e.g., max slope change) and justify it.